In [4]:
import os
import re
import base64
from pathlib import Path
from mistralai import Mistral

# 1) Load environment 
from dotenv import load_dotenv
load_dotenv()


✅ OCR text saved to bridgeport_ocr_output/bridgeport_manual_ocr.md
🖼️ Images saved to bridgeport_ocr_output/images (count: 82)


In [ ]:
# 2) Read API key
api_key = os.environ.get("MISTRAL_API_KEY")
if not api_key:
    raise RuntimeError("Missing MISTRAL_API_KEY in environment.")

client = Mistral(api_key=api_key)

In [ ]:
# Insert your inputs/outputs

pdf_path = "insert_file_path/..."
output_dir = Path("Rename_this_Output_folder")
image_folder = "..."  #name the folder for the images
images_dir = output_dir / image_folder     
md_out = output_dir / "RENAME_THIS_FINAL_MD_FILE.md"
images_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 4) Encoding in Base64 using Mistral

def encode_pdf_to_data_url(p: str) -> str:
    with open(p, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:application/pdf;base64,{b64}"

pdf_data_url = encode_pdf_to_data_url(pdf_path)

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={"type": "document_url", "document_url": pdf_data_url},
    include_image_base64=True,
)

In [ ]:
# --- Helpers for saving images & rewriting markdown ---

def safe_write_bytes(path: Path, data: bytes):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(data)

def b64_to_bytes(b64: str) -> bytes:
    if b64.startswith("data:image/"):
        b64 = b64.split(",", 1)[-1]
    return base64.b64decode(b64)

In [ ]:
# Save explicit page images (if present)

def save_page_images(pages) -> int:
    saved_names = set()
    count = 0
    for p in pages:
        imgs = getattr(p, "images", None)
        if not isinstance(imgs, list):
            continue
        for img in imgs:
            name = getattr(img, "file_name", None) or getattr(img, "name", None)
            if not name:
                ext = "png"
                mt = getattr(img, "mime_type", None)
                if isinstance(mt, str) and "/" in mt:
                    ext = (mt.split("/", 1)[-1] or "png").lower()
                name = f"img-{len(saved_names)}.{ext}"
            if name in saved_names:
                continue
            b64 = getattr(img, "image_base64", None) or getattr(img, "base64", None) or getattr(img, "content", None)
            url = getattr(img, "image_url", None)
            if b64:
                safe_write_bytes(images_dir / name, b64_to_bytes(b64))
                saved_names.add(name); count += 1
            elif isinstance(url, str) and url.startswith("data:image/"):
                safe_write_bytes(images_dir / name, b64_to_bytes(url))
                saved_names.add(name); count += 1
    return count


In [ ]:
# Rewrite ](img-0.jpeg) -> ](images/img-0.jpeg) for md navigation

REL_IMAGE_LINK_RE = re.compile(
    r'(!\[[^\]]*\]\()'                           # prefix
    r'(?!(?:https?:|data:))'                     # not http(s) or data:
    r'(?:\.\/)?'                                 # optional ./
    r'([^\s)]+\.(?:png|jpg|jpeg|webp|gif|tif|tiff|bmp|svg))'  # filename
    r'(\))',
    flags=re.IGNORECASE
)

def rewrite_rel_links(md: str) -> str:
    def _repl(m):
        pre, fname, suf = m.groups()
        # Avoid double-prepending if the filename already has the folder prefix
        if not fname.lower().startswith(f"{image_folder.lower()}/"):
            return f"{pre}{image_folder}/{fname}{suf}"
        return m.group(0)
    return REL_IMAGE_LINK_RE.sub(_repl, md)

In [ ]:
# Catch embedded data-URIs (just in case)
# Not mandatory/could be removed

DATA_URI_RE = re.compile(r'!\[([^\]]*)\]\((data:image\/[a-zA-Z0-9.+-]+;base64,[^)]+)\)')
def rewrite_md_and_save_data_uris(md: str) -> str:
    counter = 0
    def _repl(match):
        nonlocal counter
        alt = match.group(1)
        data_uri = match.group(2)
        fname = f"md_embed_{counter}.png"
        safe_write_bytes(images_dir / fname, b64_to_bytes(data_uri))
        counter += 1
        return f"![{alt}]({image_folder}/{fname})"
    return DATA_URI_RE.sub(_repl, md)


In [ ]:
# --- Save everything ---

pages = getattr(ocr_response, "pages", []) or []
images_saved = save_page_images(pages)

md_parts = []
for p in pages:
    md = getattr(p, "markdown", "") or ""
    if not md: 
        continue
    md = rewrite_md_and_save_data_uris(md)
    md = rewrite_rel_links(md)
    md_parts.append(md)

output_dir.mkdir(parents=True, exist_ok=True)
with open(md_out, "w", encoding="utf-8") as f:
    f.write("\n\n---\n\n".join(md_parts))

print(f"OCR text saved to {md_out}")
print(f"Images saved to {images_dir} (count: {images_saved})")